In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import numpy as np
import os

# ── [코랩 전용] 구글 드라이브 마운트 ──────────────────────────
# 만약 이미지가 구글 드라이브에 있다면 주석을 해제하고 연동하세요.
# from google.colab import drive
# drive.mount('/content/drive')

# ── 변수 설정 ──────────────────────────────────────────────
# 본인의 코랩 환경 내 실제 데이터 경로로 수정해 주세요.
data_dir   = "./data/archive/imgs_zip/imgs/use"
img_size   = (224, 224)
batch_size = 32
seed       = 42

# 경로 존재 여부 체크 (에러 방지용 가드)
if not os.path.exists(data_dir):
    print(f"[경고] {data_dir} 경로가 존재하지 않습니다. 실제 데이터 경로로 변경해주세요!")

# ── 데이터 로드 ────────────────────────────────────────────
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.3,
    subset="training",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size
)

val_test_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.3,
    subset="validation",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size
)

val_batches = tf.data.experimental.cardinality(val_test_ds)
val_ds  = val_test_ds.take(val_batches // 2)
test_ds = val_test_ds.skip(val_batches // 2)

class_names = train_ds.class_names
num_classes = len(class_names)
print("클래스 수:", num_classes)
print("클래스 목록:", class_names)

# ── Class Weight (F1 개선 핵심) ────────────────────────────
labels_all = []
for _, y in train_ds:
    labels_all.extend(y.numpy())

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels_all),
    y=labels_all
)
class_weight_dict = dict(enumerate(class_weights))
print("Class Weights 세팅 완료")

# ── 데이터 증강 (EfficientNet 맞춤형 정밀화) ─────────────────
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2),
    layers.RandomTranslation(0.1, 0.1),
])

AUTOTUNE = tf.data.AUTOTUNE
# 팁: 데이터 증강 후 정수형 픽셀 정보([0, 255]) 상태 그대로 유지되도록 매핑합니다.
train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y)).cache().prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)

# ── EfficientNetB0 전이학습 ─────────────────────────────────
# EfficientNetB0는 파라미터 수가 약 530만 개로 ResNet50(약 2500만 개)보다 5배 가볍지만 정확도는 동급 이상입니다.
base_model = EfficientNetB0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False  # Feature Extractor 가중치 고정

model = models.Sequential([
    layers.InputLayer(input_shape=(224, 224, 3)),
    # [중요 고침] EfficientNet 내부적으로 자체 Rescaling 레이어가 탑재되어 있어
    # 외부에서 1./255 처리를 하면 데이터가 이중 왜곡됩니다. 기존 rescaling 라인은 제거했습니다.
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

model.summary()

# ── 콜백 세팅 ─────────────────────────────────────────────
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    min_delta=0.001,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

# ── 학습 진행 ─────────────────────────────────────────────
print("\n[학습 시작] 상단 분류층(Dense Layer)만 최적화")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    class_weight=class_weight_dict,
    callbacks=[early_stopping, reduce_lr]
)

# ── 최종 테스트 평가 ────────────────────────────────────────
test_loss, test_acc = model.evaluate(test_ds)
print(f"\n[최종 결과] Test Accuracy: {test_acc:.4f} | Test Loss: {test_loss:.4f}")

# ── F1 Score & Classification Report ─────────────────────
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model(images, training=False)
    y_true.extend(labels.numpy())
    y_pred.extend(tf.argmax(preds, axis=1).numpy())

print("\n[Classification Report]")
print(classification_report(y_true, y_pred, target_names=class_names))

# ── 학습 곡선 시각화 ────────────────────────────────────────
acc      = history.history["accuracy"]
val_acc  = history.history["val_accuracy"]
loss     = history.history["loss"]
val_loss = history.history["val_loss"]

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(acc, label="Train Accuracy", color="blue")
plt.plot(val_acc, label="Val Accuracy", color="orange")
plt.title("Accuracy Curve")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(loss, label="Train Loss", color="blue")
plt.plot(val_loss, label="Val Loss", color="orange")
plt.title("Loss Curve")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# ── 모델 저장 ──────────────────────────────────────────────
model.save("car_efficientnet_model.h5")
print("모델 저장 완료: car_efficientnet_model.h5")